# FHIR-It-Will text-to-FHIR quick test

This notebook calls the local `POST /v1/NAR2FHIR` API with **synthetic data** and a caller-supplied OpenRouter key.

## Before running

1. From this directory, install dependencies: `python -m pip install -r requirements.txt`
2. Configure the repository `.env`, then provision the database and API key once: `docker compose --profile setup run --rm bootstrap`
3. Allow local OpenRouter testing in `docker-compose.override.yml`:

   ```yaml
   services:
     api:
       environment:
         ALLOW_INSECURE_TRANSPORT: "true"
         LLM_EGRESS_ALLOWLIST: openrouter.ai
   ```
4. Start the stack from the repository root: `docker compose up -d`
5. Run the cells below. Keys are requested with `getpass` and remain in kernel memory only.

Use synthetic narratives only. Generated FHIR is untrusted until the validation report is inspected.

In [3]:
# ruff: noqa: T201
import getpass
import os
import shutil
import subprocess
import time
from pathlib import Path

import httpx
from IPython.display import JSON, display

BASE_URL = os.getenv("FHIRBRIDGE_BASE", "http://localhost:8000").rstrip("/")
MODEL = os.getenv("FHIRBRIDGE_LLM_MODEL", "openai/gpt-4o-mini")

api_key = os.getenv("FHIRBRIDGE_API_KEY") or getpass.getpass("FHIR-It-Will API key (fhirb_...): ")
openrouter_key = getpass.getpass("OpenRouter API key: ")

if not api_key.strip() or not openrouter_key.strip():
    raise ValueError("Both API keys are required.")

client = httpx.Client(base_url=BASE_URL, timeout=300.0)
print(f"Configured {BASE_URL} with model {MODEL}; keys are held only in kernel memory.")

Configured http://localhost:8000 with model openai/gpt-4o-mini; keys are held only in kernel memory.


In [4]:
# Start the local Docker Compose stack, then wait for the API.
if BASE_URL.startswith(("http://localhost", "http://127.0.0.1")):
    search_from = Path.cwd().resolve()
    repo_root = next(
        (
            path
            for path in (search_from, *search_from.parents)
            if (path / "docker-compose.yml").is_file()
        ),
        None,
    )
    docker = shutil.which("docker")
    if repo_root is None:
        raise RuntimeError("Could not locate docker-compose.yml from the notebook directory.")
    if docker is None:
        raise RuntimeError("Docker is not installed or is not available on PATH.")

    # Resolve the fixed executable first and fail with an actionable Windows message.
    engine = subprocess.run(  # noqa: S603
        [docker, "info", "--format", "{{.ServerVersion}}"],
        capture_output=True,
        text=True,
        check=False,
    )
    if engine.returncode != 0:
        raise RuntimeError(
            "Docker Desktop is installed, but its engine is not running. "
            "Open Docker Desktop, wait until it reports that the engine is running, "
            "then rerun this cell."
        )

    compose = subprocess.run(  # noqa: S603
        [docker, "compose", "up", "-d"],
        cwd=repo_root,
        capture_output=True,
        text=True,
        check=False,
    )
    if compose.returncode != 0:
        raise RuntimeError(
            "Docker Compose could not start the API stack. Run `docker compose up -d` "
            "from the repository root to see the detailed Compose error."
        )

last_error = None
for _ in range(45):
    try:
        health = client.get("/livez")
        health.raise_for_status()
        break
    except (httpx.ConnectError, httpx.HTTPStatusError) as exc:
        last_error = exc
        time.sleep(2)
else:
    raise RuntimeError(f"API did not become live at {BASE_URL} within 90 seconds.") from last_error

print("API is live.")

RuntimeError: Docker Desktop is installed, but its engine is not running. Open Docker Desktop, wait until it reports that the engine is running, then rerun this cell.

In [ ]:
# Synthetic test data only. Replace it only with other synthetic data.
synthetic_text = (
    "A 62-year-old synthetic patient has blood pressure 128/82 mmHg. "
    "They take metformin 500 mg by mouth twice daily."
)

response = client.post(
    "/v1/NAR2FHIR",
    headers={
        "Authorization": f"Bearer {api_key}",
        "X-LLM-Provider": "openrouter",
        "X-LLM-Model": MODEL,
        "X-LLM-API-Key": openrouter_key,
        "X-PHI-Egress-Acknowledged": "true",
    },
    json={"text": synthetic_text},
)

if response.is_error:
    try:
        body = response.json()
        error = body.get("error", body)
        detail = error.get("message") or error.get("code") or "Request failed"
    except (ValueError, AttributeError):
        detail = "Request failed without a JSON error envelope"
    raise RuntimeError(f"NAR2FHIR returned HTTP {response.status_code}: {detail}")

result = response.json()
assert result["validated"] is False, "NAR2FHIR output must remain explicitly unvalidated"
print(f"Conversion {result['conversion_id']} completed; validation is still required.")

In [ ]:
# Inspect the generated, still-untrusted FHIR Bundle and assembly notes.
print(
    f"Provider: {result['llm']['provider']} | "
    f"Model: {result['llm']['model']} | "
    f"Qualification: {result['llm']['qualification_tier']}"
)
display(JSON(result["bundle"]))
display(JSON(result["assembly"]))

In [ ]:
# Validate the generated Bundle. A successful HTTP response is not itself a pass.
validation_response = client.post(
    "/v1/validate",
    headers={"Authorization": f"Bearer {api_key}"},
    json={"resource": result["bundle"]},
)
validation_response.raise_for_status()
report = validation_response.json()

assert len(report["layers"]) == 8, "Validation report must contain all eight layers"
print(f"Validation status: {report['status']} | Conformant: {report['conformant']}")
display(JSON(report["layers"]))

In [ ]:
# Remove credentials from kernel memory when finished.
client.close()
del openrouter_key, api_key
print("HTTP client closed and key variables removed.")